In [323]:
# from IPython.core.interactiveshell import InteractiveShell
# InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import seaborn as sns
sns.set()
import gc
gc.collect()
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import set_matplotlib_formats
from IPython.display import clear_output
set_matplotlib_formats('retina')
#from tqdm import tqdm
#tqdm.pandas()
import numpy as np
from pyhive import presto
from datetime import datetime, timedelta
# from bson import ObjectId
from functools import reduce
from sklearn.cluster import KMeans
# from pymongo import MongoClient
import glob
import warnings
warnings.filterwarnings('ignore')
from datetime import date
import json
import re
#import dtale
#from h3 import h3
# import pandasql as ps
import datetime
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose

In [414]:
metabase_connection = presto.connect(
        # host='bi-trino-2.serving.data.production.internal',
#         host='bi-trino.serving.data.production.internal',
        #host='bi-presto.serving.data.production.internal',
        #host='prime-trino.serving.data.production.internal',
         host="processing.processing.data.production.internal",
         
#         host="presto.processing.yoda.run",
        # host="processing.processing.data.production.internal",


        port=80,
        protocol='http',
        catalog ='hive',
        username='sameer.kumar@rapido.bike')

: 

In [ ]:
q="""
    with v1 as(
    select yyyymmdd as startDateOfCurrentPlan,eventprops__eventprops__userid as captain_id,eventprops__eventprops__phone_number as mobileNumber,updated_epoch as epoch,
    eventprops__eventprops__city as city,
    eventprops__eventprops__type as currentPlan,
    lead(yyyymmdd) over(partition by eventprops__eventprops__phone_number order by yyyymmdd) as endDateOfCurrentPlan,
    lead(updated_epoch) over(partition by eventprops__eventprops__phone_number order by yyyymmdd) as endEpoch,
    lead(eventprops__eventprops__type) over(partition by eventprops__eventprops__phone_number order by yyyymmdd) as planChangedTo
    from canonical.iceberg_production_data_raw_clevertap_captain_clevertap_captain_events_v2_eventname_byob_screen_immutable
    where yyyymmdd between '20250220' and '20250304'
    and eventprops__eventprops__action='plan_confirmed'
    and lower(eventprops__eventprops__city) in ('kolkata','ahmedabad','delhi','chandigarh')
),
v2 as(
 select captain_id,mobileNumber,city,currentPlan,planChangedTo,date_parse(startDateOfCurrentPlan,'%Y%m%d') as startDateOfCurrentPlan,epoch,endEpoch,
       date_parse(case when endDateOfCurrentPlan is null then '20250304' 
       else  endDateOfCurrentPlan
       end,'%Y%m%d') as endDateOfCurrentPlan
    from v1
    where planChangedTo is not null
),
v3 as (
   select *,date_diff('day',startDateOfCurrentPlan,endDateOfCurrentPlan) as numDays
   from v2
),
v4 as(
 select a.mobileNumber,
 a.captain_id,
 a.city,
 a.currentPlan,
 a.planChangedTo,
 a.startDateOfCurrentPlan,
 a.endDateOfCurrentPlan,
 a.numDays,
 count(distinct b.order_id) totalOrders
 from v3 a
 inner join orders.order_logs_fact b
 on a.captain_id=b.captain_id
 and cast(b.dropped_epoch as bigint) between cast(a.epoch as bigint) and cast(a.endEpoch as bigint)
 where order_status='dropped'
 group by 1,2,3,4,5,6,7,8
)
select *
from v4
where numDays>1

"""
captains=pd.read_sql(q,metabase_connection)


In [418]:
captains.sort_values(by=['startDateOfCurrentPlan'],inplace=True)
captains.reset_index(drop='index',inplace=True)

In [ ]:
captains['nextChangeDate']=captains.groupby(['mobileNumber'])['yyyymmdd'].shift(-1)
captains['nextChangePlan']=captains.groupby(['mobileNumber'])['plan'].shift(-1)

In [398]:
arr=captains.mobileNumber.to_list()

In [ ]:
captains[captains.mobileNumber=='9227771040']

,yyyymmdd,mobileNumber,city,plan
0,20250220,9227771040,Ahmedabad,EPKM
3248,20250227,9227771040,Ahmedabad,EPKM
3667,20250227,9227771040,Ahmedabad,Commission
3862,20250227,9227771040,Ahmedabad,EPKM


In [ ]:
captains.nextChangeDate.fillna('20250304',inplace=True)

In [ ]:
captains['nextChangeDate']=pd.to_datetime(captains['nextChangeDate'])
captains['yyyymmdd']=pd.to_datetime(captains['yyyymmdd'])

In [ ]:
captains['noOFDays']=(captains['nextChangeDate']-captains['yyyymmdd'])/pd.Timedelta(days=1)

In [ ]:
captains=captains[captains['noOFDays']>=2]

In [ ]:
captains=captains[captains['nextChangeDate']>pd.to_datetime('2025-02-27')]

In [ ]:
captains=captains[~captains['nextChangePlan'].isna()].reset_index()

In [ ]:
captains.drop(columns=['index'],inplace=True)

In [ ]:
captainIds=captains.captain_id.to_list()

In [ ]:
captains

,mobileNumber,captain_id,city,currentPlan,planChangedTo,startDateOfCurrentPlan,endDateOfCurrentPlan,numDays,totalOrders
0,9870520668,66d194bfabc9a470c6544ccd,Delhi,EPKM,EPKM,2025-02-22 00:00:00.000,2025-03-02 00:00:00.000,8,40
1,9654585854,65fc1c48dca63f6df15afd5d,Delhi,EPKM,EPKM,2025-02-28 00:00:00.000,2025-03-04 00:00:00.000,4,10
2,9599866418,62a942e68cbb359963635042,Delhi,EPKM,Subs,2025-02-26 00:00:00.000,2025-03-03 00:00:00.000,5,2
3,8527654080,661f8871cb3f070876f7c6a9,Delhi,EPKM,EPKM,2025-02-22 00:00:00.000,2025-02-28 00:00:00.000,6,79
4,9433652711,618e1538ae4f73f501166720,Kolkata,EPKM,EPKM,2025-02-24 00:00:00.000,2025-03-01 00:00:00.000,5,31
...,...,...,...,...,...,...,...,...,...
1009,7056130493,615327cdfd38d34ac38ab036,Delhi,EPKM,Subs,2025-02-26 00:00:00.000,2025-03-02 00:00:00.000,4,22
1010,8090077077,5d1a30533b752c45cf922c34,Delhi,EPKM,Commission,2025-02-22 00:00:00.000,2025-02-25 00:00:00.000,3,5
1011,7056869376,6746ee6a27f43a01f4e566b2,Delhi,EPKM,EPKM,2025-02-25 00:00:00.000,2025-03-04 00:00:00.000,7,10
1012,8178111989,616f76c7fbce661a5c9d0f2a,Delhi,EPKM,EPKM,2025-02-24 00:00:00.000,2025-03-01 00:00:00.000,5,22


In [ ]:
q="""
 select yyyymmdd,captain_id,count(distinct order_id) as numOrder
 from orders.order_logs_fact
 where lower(order_status)='dropped'
"""

In [ ]:
captain_subsription
cm_distribution